# 03 — Feature Engineering

Create clinically motivated features and prepare the modeling matrix.

**Requirements:** Section 5 (PRD), Section 4.3 (TRD)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.feature_engineering import engineer_features, engineer_longitudinal_features, get_feature_columns
from src.preprocessing import preprocess_pipeline
from src.utils import data_path, set_seed

set_seed()

In [ ]:
# Load preprocessed data (or run preprocessing inline)
merged_path = data_path('processed', 'oasis_merged_final.csv')
if merged_path.exists():
    df = pd.read_csv(merged_path)
else:
    from src.data_loader import load_and_merge
    _, _, merged = load_and_merge()
    df = preprocess_pipeline(merged)

df.head()

In [ ]:
# Engineer cross-sectional features
df = engineer_features(df)
df[['BrainAtrophyRatio', 'CognitiveReserveIndex', 'AgeGroup', 'MMSESeverity', 'MMSE_norm', 'SES_EDUC_interaction']].head()

In [ ]:
# Engineer OASIS-2 longitudinal features (if longitudinal columns present)
oasis2_path = data_path('processed', 'oasis2_cleaned.csv')
if oasis2_path.exists():
    df2 = pd.read_csv(oasis2_path)
    df2_long = engineer_longitudinal_features(df2)
    print(df2_long[['ID', 'CDR_change', 'MMSE_slope', 'ConversionFlag', 'TimeToConversion']].drop_duplicates('ID').head())

In [ ]:
# Build modeling feature matrix
feature_cols = get_feature_columns(include_engineered=True)
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df['target']
print(f'Feature matrix: {X.shape} | Target: {y.value_counts().to_dict()}')

In [ ]:
# Save feature-engineered dataset
output_path = data_path('processed', 'oasis_merged_final.csv')
df.to_csv(output_path, index=False)
print(f'Saved feature-engineered dataset to {output_path}')